<a name="top"></a><img src="../images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 4.2：FIRRTL AST 遍历

**上一节：[FIRRTL 介绍](4.1_firrtl_ast.ipynb)**<br>
**下一节：[常见 Pass 惯用法](4.3_firrtl_common_idioms.ipynb)**

### 理解 IR 节点的子节点

编写 Firrtl pass 通常需要编写遍历 Firrtl 数据结构的函数，要么收集信息，要么用新的 IR 节点替换 IR 节点。

IR 数据结构是一棵树，其中每个 IR 节点可以有若干个子节点（而这些子节点又可以有更多的子节点，依此类推）。没有子节点的 IR 节点称为叶子节点。

不同的 IR 节点可以有不同的子节点类型。下表显示了每种 IR 节点类型可能的子节点类型：

```
+------------+-----------------------------+
|    Node    |          Children           |
+------------+-----------------------------+
| Circuit    | DefModule                   |
| DefModule  | Port, Statement             |
| Port       | Type, Direction             |
| Statement  | Statement, Expression, Type |
| Expression | Expression, Type            |
| Type       | Type, Width                 |
| Width      |                             |
| Direction  |                             |
+------------+-----------------------------+
```

### map 函数

要编写一个遍历 `Circuit` 的函数，我们首先需要理解函数式编程概念 `map`。

#### 理解 Seq.map
一个 Scala 字符串序列可以表示为一棵树，根节点为 `Seq`，子节点为 `"a"`、`"b"` 和 `"c"`：
```scala
val s = Seq("a", "b", "c")
```
```
    Seq
 /   |   \
"a" "b" "c"
```

假设我们定义一个函数 `f`，给定一个字符串参数 `x`，将 `x` 与自身连接：
```scala
def f(x: String): String = x + x
```

我们可以调用 `s.map` 返回一个新的 `Seq[String]`，其子节点是将 `f` 应用于 s 的每个子节点的结果：
```scala
val s = Seq("a", "b", "c")
def f(x: String): String = x + x  // 重复声明以保持清晰
val t = s.map(f)
println(t) // Seq("aa", "bb", "cc")
```
```
     Seq
 /    |    \
"aa" "bb" "cc"
```

#### 理解 Firrtl 的 map

我们使用这种"映射"思想，在 IR 节点上创建我们自己的自定义 `map` 方法。假设我们有一个表示 1 + 1 的 `DoPrim` 表达式；这可以描绘成一个表达式树，根节点为 `DoPrim`：
```
        DoPrim
     /          \
UIntValue    UIntValue
```

如果我们有一个函数 `f`，它接受一个 `Expression` 参数并返回一个新的 `Expression`，我们可以将其"映射"到给定 IR 节点的所有子 `Expression` 上，比如我们的 `DoPrim`。这将返回以下新的 `DoPrim`，其子节点是将 `f` 应用于 `DoPrim` 的每个 `Expression` 子节点的结果：
```
        DoPrim
     /          \
f(UIntValue)    f(UIntValue)
```

有时 IR 节点有多个类型的子节点。例如，`Conditionally` 同时具有 `Expression` 和 `Statement` 子节点。在这种情况下，map 只会将其函数应用于子节点中与函数参数类型（以及返回值类型）匹配的那些：
```scala
val c = Conditionally(info, e, s1, s2) // e: Expression, s1, s2: Statement, info: FileInfo
def fExp(e: Expression): Expression = ...
def fStmt(s: Statement): Statement = ...
c.map(fExp)  // Conditionally(fExp(e), s1, s2)
c.map(fStmt) // Conditionally(e, fStmt(s1), fStmt(s2))
```

Scala 有"中缀表示法"，允许你在调用只有一个参数的函数时省略 `.` 和括号。通常，我们使用中缀表示法编写这些 map 函数：
```scala
c map fExp  // 等价于 c.map(fExp)
c map fStmt // 等价于 c.map(fStmt)
```

### 前序遍历

要遍历 Firrtl 树，我们使用 `map` 编写递归函数，访问我们关心的每个节点的每个子节点。

假设我们想收集设计中声明的每个寄存器的名称；我们知道这需要访问每个 `Statement`。然而，一些 `Statement` 节点可以有子 `Statement`。因此，我们需要编写一个函数，它既会检查其输入参数是否为 `DefRegister`，如果不是，则会递归地将 `f` 应用于其输入参数的所有 `Statement` 子节点：

下面的函数 `f` 与我们描述的函数类似，但它接受两个参数：一个可变的寄存器名称哈希集合，和一个 `Statement`。使用函数柯里化，我们可以只传递第一个参数，以返回一个具有所需类型签名（`Statement=>Statement`）的新函数：

```scala
def f(regNames: mutable.HashSet[String]())(s: Statement): Statement = s match {
  // 如果是寄存器，将名称添加到 regNames
  case r: DefRegister =>
    regNames += r.name
    r // 返回参数不变（没问题，因为 DefRegister 没有 Statement 子节点）
  // 如果不是，则将 f(regNames) 应用于所有子 Statement
  case _ => s map f(regNames) // 注意 f(regNames) 的类型是 Statement=>Statement
}
```

这种模式在 Firrtl 中非常常见，被称为"前序遍历"，因为递归函数在递归应用于其子节点之前匹配原始 IR 节点。

### 后序遍历

我们可以将前面的示例写成"后序遍历"形式，如下所示：

```scala
def f(regNames: mutable.HashSet[String]())(s: Statement): Statement = 
  // 注意我们先立即递归到子节点，然后再匹配
  s map f(regName) match {
    // 如果是寄存器，将名称添加到 regNames
    case r: DefRegister =>
      regNames += r.name
      r // 返回参数不变（没问题，因为 DefRegister 没有 Statement 子节点）
    // 如果不是，返回 s
    case _ => s // 注意 s 的所有 Statement 子节点已经应用了 f(regNames)
  }
```

虽然这两个示例的遍历顺序不同，但对于这个用例（以及许多其他用例）没有区别。然而，当遍历顺序重要时，这是一个需要牢记的重要工具。